In [ ]:
def assess_testability(day: str, latitude: float, longitude: float) -> Dict[str, Any]:
    """Return an assessment whether it's a good day to test-drive a used car at the given date and location."""
    w = get_weather_on_date(latitude, longitude, day)
    if not w.get('success'):
        return { 'success': False, 'error': 'Could not retrieve weather: ' + str(w.get('error')) }
    summary = w.get('summary', {})
    reasons = []
    score = 0

    prec = summary.get('precip_total_mm', 0)
    if prec and prec > 0.5:
        reasons.append(f'Precipitation total {prec} mm — not ideal for testing')
        score -= 2

    wind = summary.get('wind_avg_kmh', 0)
    if wind and wind > 40:
        reasons.append(f'High wind average {wind:.1f} km/h — caution')
        score -= 1

    tmin = summary.get('temp_min')
    tmax = summary.get('temp_max')
    if tmin is not None and tmin < -5:
        reasons.append(f'Low temperature {tmin}°C — may affect battery/startup')
        score -= 1
    if tmax is not None and tmax > 40:
        reasons.append(f'High temperature {tmax}°C — caution with engines/AC')
        score -= 1

    recommendation = 'Bien' if score >= 0 else 'No recomendable'
    return { 'success': True, 'recommendation': recommendation, 'reasons': reasons, 'score': score, 'summary': summary }

def check_vehicle_safety(make: str, model: str, year: int, vin: str = None) -> Dict[str, Any]:
    """Check recalls and safety ratings for a vehicle using NHTSA APIs. Returns a dict with potentially partial results if some calls fail."""
    try:
        recalls = []
        recalls_error = None
        try:
            base_url = 'https://api.nhtsa.gov/recalls/recallsByVehicle'
            params = {'make': make, 'model': model, 'modelYear': year}
            r = httpx.get(base_url, params=params, timeout=10.0)
            r.raise_for_status()
            recalls_data = r.json() if r.text else {}
            for rec in recalls_data.get('results', []):
                recalls.append({
                    'campaign_number': rec.get('NHTSACampaignNumber') or rec.get('CampaignNumber'),
                    'component': rec.get('Component'),
                    'summary': rec.get('Summary') or rec.get('RecallSummary'),
                    'consequence': rec.get('Consequence') or rec.get('Conequence'),
                    'remedy': rec.get('Remedy'),
                    'date': rec.get('ReportReceivedDate') or rec.get('Date'),
                })
        except Exception as e_rec:
            recalls = []
            recalls_error = str(e_rec)

        vin_info = None
        if vin:
            try:
                vin_url = f'https://api.nhtsa.gov/vehicles/DecodeVin/{vin}'
                r = httpx.get(vin_url, timeout=10.0)
                r.raise_for_status()
                vin_json = r.json() if r.text else None
                if vin_json and isinstance(vin_json, dict):
                    results = vin_json.get('Results') or vin_json.get('results')
                    if results:
                        vin_info = results[0]
            except Exception:
                vin_info = None

        safety_ratings = {}
        try:
            safety_url = f'https://api.nhtsa.gov/SafetyRatings/vehicle/{year}/{make}/{model}'
            r = httpx.get(safety_url, timeout=10.0)
            r.raise_for_status()
            safety_json = r.json() if r.text else None
            if safety_json and isinstance(safety_json, dict):
                results = safety_json.get('results') or safety_json.get('Results')
                if results:
                    first = results[0]
                    safety_ratings = {
                        'overall_rating': first.get('OverallRating'),
                        'frontal_crash': first.get('FrontalCrashRating'),
                        'side_crash': first.get('SideCrashRating'),
                        'rollover': first.get('RolloverRating'),
                    }
        except Exception:
            safety_ratings = {}

        vehicle_info = {
            'make': make,
            'model': model,
            'year': year,
            'recalls': recalls,
            'total_recalls': len(recalls),
            'safety_ratings': safety_ratings
        }
        if vin_info:
            vehicle_info['vin_details'] = {
                'manufacturer': vin_info.get('Manufacturer'),
                'plant': vin_info.get('PlantCity'),
                'body_class': vin_info.get('BodyClass'),
                'fuel_type': vin_info.get('FuelTypePrimary'),
                'engine': vin_info.get('EngineConfiguration'),
                'transmission': vin_info.get('TransmissionStyle'),
            }

        recommendations = []
        if recalls:
            recommendations.append('Verificar todos los recalls abiertos antes de comprar')
            recommendations.append('Solicitar documentación de reparaciones de recalls previos')
        if safety_ratings and safety_ratings.get('overall_rating'):
            recommendations.append('Considerar calificaciones de seguridad en la decisión de compra')

        out = {'success': True, 'data': vehicle_info, 'recommendations': recommendations}
        if recalls_error:
            out['recalls_error'] = recalls_error
        return out
    except Exception as e:
        return {'success': False, 'error': str(e)}